### Transformar Datos "Employees" - Expandir Arrays
1. Acceder a los elementos del objeto JSON
2. Eliminar elementos duplicados del array
3. Anular el anidamiento del array
4. Asignar al campo "status" los siguientes valores descriptivos:<br>
(0: Inactive, 1: Active)
5. Escribir los datos "transformados" en el schema "silver"

In [0]:
SELECT *
FROM zenviro.silver.employees_json;

#### 1. Acceder a los elementos del objeto JSON
> `<column_name.object>`

In [0]:
SELECT json_value.employee_id,
        json_value.department_id,
        json_value.job_position_id,
        json_value.name,
        json_value.gender,
        json_value.birth_date,
        json_value.hire_date,
        json_value.email,
        json_value.phone,
        json_value.status
FROM zenviro.silver.employees_json;

#### 2. Eliminar elementos duplicados del array
[Doc - Función "array_distinct"](https://learn.microsoft.com/es-es/azure/databricks/sql/language-manual/functions/array_distinct)

In [0]:
SELECT json_value.employee_id,
        json_value.department_id,
        json_value.job_position_id,
        array_distinct(json_value.name) as name,
        json_value.gender,
        json_value.birth_date,
        json_value.hire_date,
        json_value.email,
        json_value.phone,
        json_value.status
FROM zenviro.silver.employees_json;

#### 3. Anular el anidamiento del array
[Doc - Función "explode"](https://learn.microsoft.com/es-es/azure/databricks/sql/language-manual/functions/explode)

In [0]:
CREATE OR REPLACE TEMP VIEW tv_employees_explode
AS
  SELECT json_value.employee_id,
          json_value.department_id,
          json_value.job_position_id,
          explode(array_distinct(json_value.name)) as name,
          json_value.gender,
          json_value.birth_date,
          json_value.hire_date,
          json_value.email,
          json_value.phone,
          json_value.status
  FROM zenviro.silver.employees_json;

In [0]:
SELECT * FROM tv_employees_explode;

In [0]:
SELECT employee_id,
       department_id,
       job_position_id,
       name.first_name,
       name.last_name,
       gender,
       birth_date,
       hire_date,
       email,
       phone,
       status
FROM tv_employees_explode

#### 4. Asignar al campo "status" los siguientes valores descriptivos:<br>
(0: Inactive, 1: Active)

In [0]:
SELECT employee_id,
       department_id,
       job_position_id,
       name.first_name,
       name.last_name,
       gender,
       CAST(birth_date AS DATE) AS birth_date,
       CAST(hire_date AS DATE) AS hire_date,
       email,
       phone,
       CASE status
        WHEN 0 THEN 'Inactive'
        WHEN 1 THEN 'Active'
       END AS status
FROM tv_employees_explode

#### 5. Escribir los datos "transformados" en el schema "silver"

In [0]:
CREATE TABLE IF NOT EXISTS zenviro.silver.employees
AS
  SELECT employee_id,
        department_id,
        job_position_id,
        name.first_name,
        name.last_name,
        gender,
        CAST(birth_date AS DATE) AS birth_date,
        CAST(hire_date AS DATE) AS hire_date,
        email,
        phone,
        CASE status
          WHEN 0 THEN 'Inactive'
          WHEN 1 THEN 'Active'
        END AS status
  FROM tv_employees_explode;

In [0]:
SELECT * FROM zenviro.silver.employees;